# HappyCity Gen : entraînement sur Kaggle

Entraîne les variantes **A, B, C** et les **ablations** de C avec la même commande que
la CLI locale (`happycity train …`). Le notebook ne contient aucune logique d'entraînement.

**Prérequis**
1. Accélérateur **GPU** (T4 ou P100) et **Internet** activés dans les paramètres.
2. Un dataset Kaggle contenant les fichiers produits localement par
   `happycity make-dataset`, à savoir `data/processed/dataset/*.npz` (environ 7 Mo). Le
   notebook les retrouve où qu'ils soient sous `/kaggle/input`.
3. Le dépôt est **public** : un simple `git clone` suffit, sans jeton.

Les sorties (journaux, checkpoints, configuration résolue, commit git) sont écrites dans
`/kaggle/working/experiments` et compressées à la fin.

In [ ]:
# --- À renseigner
REPO_URL = "https://github.com/soufianebouragaa01/happycity-gen.git"  # dépôt public
REF = "main"            # branche, tag ou commit (pour la reproductibilité)
EPOCHS = 40             # 40 = configuration par défaut
SEED = 42
RUNS = [                # (variante, ablation)
    ("A", "none"),
    ("B", "none"),
    ("C", "none"),
    ("C", "no_modulation"),
    ("C", "no_attention"),
    ("C", "no_projection"),
    ("C", "no_diversity"),
]

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/working/happycity-gen")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REF], check=True)
# torch, numpy, pandas, scipy sont fournis par Kaggle : on n'installe que le reste,
# sans réinstaller torch (--no-deps pour le paquet lui-même).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hydra-core>=1.3", "omegaconf>=2.3", "typer>=0.12"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--ignore-requires-python", "-e", str(REPO)], check=True)
print(subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
found = sorted(Path("/kaggle/input").rglob("train.npz"))
assert found, "train.npz introuvable : ajoutez le dataset produit par `happycity make-dataset`"
DATASET_DIR = found[0].parent
print("jeu de données :", DATASET_DIR, sorted(p.name for p in DATASET_DIR.glob("*.npz")))

In [ ]:
import os

os.chdir(REPO)
os.environ["HAPPYCITY_CONFIG_DIR"] = str(REPO / "configs")
for model, ablation in RUNS:
    cmd = ["happycity", "train", f"model={model}", f"ablation={ablation}", "paths=kaggle",
           f"paths.dataset_dir={DATASET_DIR}", f"train.epochs={EPOCHS}", f"train.seed={SEED}"]
    print("▶", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(chr(10).join(proc.stdout.splitlines()[-3:] + proc.stderr.splitlines()[-15:]))
    if proc.returncode != 0:
        print(f"✗ échec de {model}-{ablation} (code {proc.returncode})")

In [ ]:
import pandas as pd

EXP = Path("/kaggle/working/experiments")
rows = []
for run in sorted(EXP.iterdir()):
    log = run / "val_log.csv"
    if log.exists():
        val = pd.read_csv(log)
        sel = val[val["split"] == "val_synth"]
        best = sel.loc[sel["score"].idxmin()]
        real = val[(val["split"] == "val") & (val["epoch"] == best["epoch"])]
        rows.append({"exécution": run.name, "meilleure époque": int(best["epoch"]),
                     "score val_synth": best["score"], "|Â−a| projeté (val_synth)": best["a_err_proj"],
                     "violations brutes": best["viol_raw"], "diversité": best["diversity"],
                     "|Â−a| projeté (val réel)": float(real["a_err_proj"].iloc[0]) if len(real) else None})
summary = pd.DataFrame(rows)
summary.to_csv(EXP / "summary_validation.csv", index=False)
summary.round(4)

In [ ]:
import shutil

archive = shutil.make_archive("/kaggle/working/experiments", "zip", EXP)
print("archive :", archive, f"{Path(archive).stat().st_size / 1e6:.1f} Mo")

Les métriques ci-dessus sont celles de la **validation** (A400 doux, rapide). L'évaluation
finale, avec les métriques exactes par Dijkstra, les références, les cibles inatteignables,
`test_synth` et le test réel, se fait avec `happycity evaluate` (phase 6) sur les checkpoints
`best.pt` téléchargés.